In [2]:
# Ammar Wildan bin Mustafar (SW01083734)
# Muhammad Adam Zharfan bin Mohd Khairul Hisham (SW01083748)
# Lab Assignment 1: Web Scraping Reviews from Lazada

import time
import csv
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

def scrape_page(soup, reviews_list):
#parses the HTML content (soup) to find and extract specific review details: Reviewer Name, Date, and Review Content.It appends these details as a dictionary to the provided list.

    # Find all individual review blocks on the page
    review_blocks = soup.find_all('div', class_='review-item')
    
    for block in review_blocks:
        # Extract name, date, and text with fallbacks if elements are missing
        name = block.find('div', class_='new-buyer-name')
        date = block.find('div', class_='new-sku-info')
        content = block.find('div', class_='LinesEllipsis')
        
        reviews_list.append({
            'Author': name.get_text(strip=True) if name else 'Anonymous',
            'Date': date.get_text(strip=True) if date else 'N/A',
            'Text': content.get_text(strip=True) if content else 'No content'
        })

def scrape_all_pages(url, scroll_limit=5):
#initializes the Selenium WebDriver to handle JavaScript rendering. It navigates to the URL and performs scrolls to trigger 'Lazy Loading' of reviews up to the specified limit.

    # Initialize Chrome with automatic driver management
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()))
    all_data = []
    
    try:
        driver.get(url)
        # Brief pause to allow the user to manually solve any captcha/login popups
        print("Browser opened. Please solve any CAPTCHAs manually if they appear.")
        time.sleep(8) 

        for i in range(scroll_limit):
            print(f"Executing scroll/load step {i+1}...")
            
            # Wait for the review container to appear in the DOM
            WebDriverWait(driver, 15).until(
                EC.presence_of_element_located((By.CLASS_NAME, 'review-item'))
            )
            
            # Scroll to the bottom of the page to load more reviews
            driver.execute_script('window.scrollTo(0, document.body.scrollHeight);')
            time.sleep(3) # Wait for JS to render new content
            
            # Capture the updated HTML and pass it to the parsing function
            current_soup = BeautifulSoup(driver.page_source, 'html.parser')
            scrape_page(current_soup, all_data)
            
        return all_data
            
    finally:
        driver.quit()

def save_data_to_csv(data, filename):
#takes the list of extracted review dictionaries and saves them into a CSV file with the required headers.

    # Remove duplicates that may have been captured during scrolling
    unique_data = [dict(t) for t in {tuple(d.items()) for d in data}]
    
    with open(filename, 'w', newline='', encoding='utf-8-sig') as csvfile:
        fieldnames = ['Author', 'Date', 'Text']
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        
        writer.writeheader()
        writer.writerows(unique_data)
    print(f"Successfully saved {len(unique_data)} unique reviews to {filename}.")

#url
target_url = 'https://www.lazada.com.my/wow/gcp/my/review/product-reviews?itemId=3697872356&skuId=22842128123&spm=a2o4k.pdp_revamp.pdp_top_tab.rating_and_review'
output_file = 'ExtractedLazadaReview.csv'

#scrape
raw_reviews = scrape_all_pages(target_url, scroll_limit=5)

#save to CSV
if raw_reviews:
    save_data_to_csv(raw_reviews, output_file)
else:
    print("No reviews were extracted. Please check the URL or page status.")

Browser opened. Please solve any CAPTCHAs manually if they appear.
Executing scroll/load step 1...
Executing scroll/load step 2...
Executing scroll/load step 3...
Executing scroll/load step 4...
Executing scroll/load step 5...
Successfully saved 60 unique reviews to ExtractedLazadaReview.csv.
